# Fashion Retail Store — Data Exploration

Explorative Analyse der 7 Rohdatensätze eines Fashion-Retail-Datensatzes (`campaigns`, `channels`, `customers`, `products`, `sales`, `sales_items`, `stock`) als erster Schritt des `lakehouse-retail-pipeline`-Portfolio-Projekts.

**Inhalt:**
1. Daten laden
2. Data-Quality-Check (Rohdaten)
3. Cleaning (pro Tabelle, vor dem Merge)
4. Merge zu `transaktion_data` (Item-Ebene)
5. Integritätsprüfung
6. Stock-Coverage & finale Spaltenauswahl

In [13]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 1. Daten laden

Alle 7 CSVs aus `Data/` werden über eine gemeinsame `load_csv()`-Funktion eingelesen, Datumsspalten direkt beim Laden geparst.

In [4]:
from pathlib import Path
import pandas as pd




Path(__file__).resolve().parents[2] / "Data"

NameError: name '__file__' is not defined

In [5]:
Path.cwd()

PosixPath('/home/hicham/Dokumente/Projekt_Hicham_2025/lakehouse-retail-pipeline/notebooks')

In [6]:
DATA_DIR = Path.cwd().parent / "Data"

def load_csv(name, parse_dates=None):
    """Lädt eine der 7 Fashion-Store-CSVs aus Data/."""
    return pd.read_csv(DATA_DIR / f"dataset_fashion_store_{name}.csv", parse_dates=parse_dates)

In [7]:
list(DATA_DIR.iterdir())


[PosixPath('/home/hicham/Dokumente/Projekt_Hicham_2025/lakehouse-retail-pipeline/Data/dataset_fashion_store_channels.csv'),
 PosixPath('/home/hicham/Dokumente/Projekt_Hicham_2025/lakehouse-retail-pipeline/Data/dataset_fashion_store_campaigns.csv'),
 PosixPath('/home/hicham/Dokumente/Projekt_Hicham_2025/lakehouse-retail-pipeline/Data/dataset_fashion_store_salesitems.csv'),
 PosixPath('/home/hicham/Dokumente/Projekt_Hicham_2025/lakehouse-retail-pipeline/Data/dataset_fashion_store_stock.csv'),
 PosixPath('/home/hicham/Dokumente/Projekt_Hicham_2025/lakehouse-retail-pipeline/Data/dataset_fashion_store_sales.csv'),
 PosixPath('/home/hicham/Dokumente/Projekt_Hicham_2025/lakehouse-retail-pipeline/Data/dataset_fashion_store_products.csv'),
 PosixPath('/home/hicham/Dokumente/Projekt_Hicham_2025/lakehouse-retail-pipeline/Data/dataset_fashion_store_customers.csv')]

In [8]:
campaigns = load_csv("campaigns", parse_dates=["start_date", "end_date"])
channels = load_csv("channels")
customers = load_csv("customers", parse_dates=["signup_date"])
products = load_csv("products")
sales = load_csv("sales", parse_dates=["sale_date"])
sales_items = load_csv("salesitems", parse_dates=["sale_date"])
stock = load_csv("stock")
{name: df.shape for name, df in {
    "campaigns": campaigns, "channels": channels, "customers": customers,
    "products": products, "sales": sales, "sales_items": sales_items, "stock": stock
}.items()}

{'campaigns': (7, 7),
 'channels': (2, 2),
 'customers': (1000, 4),
 'products': (500, 9),
 'sales': (905, 7),
 'sales_items': (2253, 13),
 'stock': (1000, 3)}

## 2. Data-Quality-Check (Rohdaten)

Prüfung auf fehlende Werte je Tabelle, bevor irgendetwas gemerged oder bereinigt wird.

In [ ]:
dataframes = {
    "campaigns": campaigns, "channels": channels, "customers": customers,
    "products": products, "sales": sales, "sales_items": sales_items, "stock": stock
}

for name, df in dataframes.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if nulls.empty:
        print(f"No null values found in {name}.")
    else:
        for col, count in nulls.items():
            print(f"Column '{col}' has {count} null values in {name}.")

## 3. Cleaning (pro Tabelle, vor dem Merge)

Nur Formatierungs-Fixes, die einzelne Rohtabellen betreffen — kein Merge-Kontext nötig, deshalb hier und nicht erst nach dem Join. Entspricht dem `clean`-Task in der geplanten Airflow-Pipeline.

In [ ]:
# discount_percent von String ("10.00%") zu float (10.0) — sonst nicht sortier-/rechenbar
sales_items['discount_percent'] = (
    sales_items['discount_percent'].str.rstrip('%').astype(float))

## 4. Merge zu `transaktion_data` (Item-Ebene)

`sales` (Bestellkopf) wird mit `customers` (`customer_id`), `sales_items` (`sale_id` — 1:n, erzeugt die Item-Ebene), `products` (`product_id`), `channels` (`channel`) und `stock` (`product_id` + `country`) verknüpft. `campaigns` wird bewusst **nicht** gemerged — es gibt keinen Fremdschlüssel, `channel` bedeutet dort etwas anderes (Marketing- statt Vertriebskanal), siehe Diskussion unten bei `channel_campaigns`.

In [ ]:
transaktion_data = sales.merge(customers[['customer_id', 'age_range', 'signup_date']], on="customer_id", how="left")\
    .merge(sales_items[['item_id', 'sale_id','product_id', 'quantity', 'original_price','unit_price',
                        'discount_applied', 'discount_percent', 'item_total',  'channel_campaigns']], on="sale_id", how="left")\
    .merge(products[['product_id', 'product_name', 'category', 'brand', 'color', 'size']], on="product_id", how="left")\
    .merge(channels, on="channel", how="left")\
    .merge(stock, on=["product_id", "country"], how="left")
    
transaktion_data.head()   

,sale_id,channel,discounted,total_amount,sale_date,customer_id,country,age_range,signup_date,item_id,product_id,quantity,original_price,unit_price,discount_applied,discount_percent,item_total,channel_campaigns,product_name,category,brand,color,size,description,stock_quantity
0,10,E-commerce,0,299.70,2025-05-21,195,France,56-65,2025-02-28,2423,347,1,59.92,59.92,0.0,0.0,59.92,Website Banner,Tailored Sleeveless Tee,T-Shirts,Tiva,Blue,XS,Official online store,34.0
1,10,E-commerce,0,299.70,2025-05-21,195,France,56-65,2025-02-28,1837,218,2,48.73,48.73,0.0,0.0,97.46,Website Banner,Polished Satin Tee,T-Shirts,Tiva,Red,S,Official online store,41.0
2,10,E-commerce,0,299.70,2025-05-21,195,France,56-65,2025-02-28,2582,254,3,47.44,47.44,0.0,0.0,142.32,Website Banner,Relaxed Silk Shoes,Shoes,Tiva,Black,35,Official online store,59.0
3,100,App Mobile,0,681.05,2025-04-21,518,Germany,46-55,2025-03-29,1126,445,5,52.28,52.28,0.0,0.0,261.40,App Mobile,Bold Cotton Set,Sleepwear,Tiva,White,S,Brand mobile app,1.0
4,100,App Mobile,0,681.05,2025-04-21,518,Germany,46-55,2025-03-29,366,419,5,50.34,50.34,0.0,0.0,251.70,App Mobile,Elegant Crew Dress,Dresses,Tiva,Blue,L,Brand mobile app,1.0


## 5. Integritätsprüfung

Beweis, dass der 1:n-Merge über `sale_id` keine Zeilen verloren oder dupliziert hat: Summe von `item_total` je `sale_id` muss `sales.total_amount` entsprechen.

In [ ]:
check = transaktion_data.groupby("sale_id")["item_total"].sum().round(2)
expected = sales.set_index("sale_id")["total_amount"].round(2)
mismatch = (check - expected).abs() > 0.01

print(f"Anzahl abweichender sale_id: {mismatch.sum()} von {len(expected)}")


Anzahl abweichender sale_id: 0 von 905


## 6. Stock-Coverage & finale Spaltenauswahl

In [ ]:
# stock_quantity fehlt für 4 von 6 Ländern (stock.csv deckt nur FR/DE ab) — has_stock_data macht das explizit statt NaN stillschweigend zu füllen
print(transaktion_data[['country',"stock_quantity"]].isna().sum())
transaktion_data['has_stock_data'] = transaktion_data['stock_quantity'].notna()
transaktion_data = transaktion_data[['customer_id', 'product_id',  'product_name', 'item_total',
        'quantity', 'original_price', 'unit_price','sale_id', 'channel', 'discounted', 
        'brand','size', 'description', 'stock_quantity', 'has_stock_data',
        'country', 'signup_date', 'sale_date','category', 'item_id',
        'discount_applied', 'discount_percent', 'channel_campaigns',  'color', 'age_range']]


transaktion_data.head()   

In [1]:
import pathlib as pl
import pandas as pd

BRONZE_DIR= pl.Path("/home/hicham/Dokumente/Projekt_Hicham_2025/lakehouse-retail-pipeline/data_lake/bronze")

TABLES = [
    "campaigns", "channels", "customers"]

result = {name: pd.read_parquet(BRONZE_DIR / f"{name}.parquet") for name in TABLES}



In [2]:
result.keys()

dict_keys(['campaigns', 'channels', 'customers'])